# IP-Layer Characterization

This notebook helps you to:

1. Load CSV with characterization of web agent across TLS and IP layers.
2. Display figures and precise number not included in the paper.
3. Reproduice table 4 of section 6.1.


In [1]:
import pandas as pd

# Better dataframe display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
df = pd.read_csv("./active_tests_with_TLS_IP_layers.csv")

df["Web Agent"] = df["Web Agent"].str.strip()
df["Agent Version"] = df["Agent Version"].astype(str).str.strip()

In [3]:
# ============================================================
# Column selection for Table 4
# ============================================================

columns_of_interest = [
    "Web Agent", 'ip_asn_name_log', 'ip_is_abuser_log','ip_is_hosting_log'
]

columns_existing = [c for c in columns_of_interest if c in df.columns]

df_reduced = df[columns_existing].copy()

## Table 4

This cell reproduces Table 4:

| Tool | #ASN | IP Hosting | IP Abuser |
| ---- | ---: | ---------: | --------: |

Definitions:

* **#ASN** = number of distinct Autonomous System Numbers (ASNs) observed for the tool.
* **IP Hosting** = proportion of IP addresses classified as hosting, cloud, or data-center infrastructure.
* **IP Abuser** = proportion of IP addresses flagged by public IP reputation services as potentially abusive or suspicious.

In [4]:
# ------------------------------------------------------------
# Normalize tool names
# ------------------------------------------------------------

def normalize_tool(row):

    web_agent = str(row["Web Agent"]).strip()
    location = str(row["Agent LOCAL/CLOUD"]).upper()

    # Differentiate BrowserUse local/cloud
    if "browseruse" in web_agent.lower() and not "stealth" in web_agent.lower():

        if location == "LOCAL":
            return f"{web_agent} (Local)"

        elif location == "CLOUD":
            return f"{web_agent} (Cloud)"

    return web_agent


df["TOOL"] = df.apply(normalize_tool, axis=1)


# ============================================================
# Aggregate infrastructure metrics
# ============================================================

infra_table = (
    df.groupby("TOOL")
      .agg({
          "ip_asn_name_log": pd.Series.nunique,
          "ip_is_hosting_log": "mean",
          "ip_is_abuser_log": "mean"
      })
      .rename(columns={
          "ip_asn_name_log": "#ASN",
          "ip_is_hosting_log": "IP Hosting",
          "ip_is_abuser_log": "IP Abuser"
      })
)

# ------------------------------------------------------------
# Round float values
# ------------------------------------------------------------

infra_table = infra_table.round({
    "IP Hosting": 2,
    "IP Abuser": 2
})

# ------------------------------------------------------------
# Sort by ASN count
# ------------------------------------------------------------

infra_table = infra_table.sort_values(
    "#ASN",
    ascending=False
)

infra_table

,#ASN,IP Hosting,IP Abuser
TOOL,,,
BrowserUse (Cloud),35,0.02,0.02
Skyvern,34,0.01,0.00
BrowserUse_Stealth,3,0.98,0.98
Crawl4AI_Undetected_Browser,2,0.00,0.00
Crawl4AI_Stealth,2,0.00,0.00
Crawl4AI,2,0.00,0.00
Human,2,0.00,0.00
Antropic Claude for Chrome,1,0.00,0.00
ChatGPT Agent,1,0.88,0.14
